# 06. LOT 240058 · DP590

## CP가 먼저 낮아진 뒤 온도·출력이 함께 감소

관측: **2024-06-09 05:26:00 ~ 2024-06-09 09:02:53**

1. CP의 첫 일시 저하와 끝까지 낮게 유지된 시각은 같은가?
2. 온도·출력의 지속 하강은 CP와 동시에 시작되는가?
3. 다음 날 정기수리 기록이 준비 명령이나 누설을 입증하는가?

## 읽기 전에


원본 온도·조업이벤트·정비이력·정기수리·부품 CSV에서 독립 실행한다. 다른 노트북이나 가공 CSV를 먼저 실행할 필요가 없다. 센서마다 1분 내 유효 관측이 30개 이상일 때만 평균을 사용하고, 부족한 분과 LOT 사이 공백은 보간하지 않는다. 원시 그림도 누락 시각을 이어 붙이지 않는다.

자료는 교육용 재구성 데이터다. 데이터사전에서 온도 ℃와 출력 % 및 공정 구역 배정은 **가정**이고, CP-Z4의 실제 단위는 **확인되지 않았다**. OP의 0은 실제 제어출력으로 취급한다. CP-Z4M은 0이 아니라 작은 스케일의 중복 정보이므로 이 검토의 초점 센서에서 제외했다. 강종에 따른 출력 수준 이동도 검증된 실제 제철소 설정값을 뜻하지 않는다.

표의 기준은 원시 관측과 1분 평균을 구분해 표시했다. 표의 13:29는 13:29:00~59 구간의 평균이며, 곡선은 그 분의 가운데(13:29:30)에 표시한다. 해당 분이 끝난 뒤 계산할 수 있는 값이다. 아래 본문 표는 이번 원자료 실행에서 계산한 값이며, 원자료가 바뀌면 코드의 재실행 결과를 다시 확인한다. 이 노트북은 고장 정답을 만들거나 모델을 학습하지 않는다.

In [ ]:
LOT = 240058
NOTEBOOK_NAME = '06_LOT_240058'
PLOT_TITLE = 'CP가 먼저 낮아진 뒤 온도·출력이 함께 감소'
FOCUS = ['CP-Z4', 'TC-Z4', 'OP-Z4', 'TC-Z7', 'TC-Z8']
COMPARE = ['CP-Z4', 'OP-Z4']
ZOOM_START = '2024-06-09 07:25'
ZOOM_END = '2024-06-09 09:02:53'
extra_tables = []

# 팀 공통 경로·인코딩·한글 폰트 설정입니다.
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams
import matplotlib.dates as mdates
from IPython.display import display

# 이 하위 폴더, notebooks 폴더, 저장소 루트에서 모두 실행할 수 있습니다.
for candidate in [os.path.join('..', '..', 'data'), os.path.join('..', 'data'), 'data']:
    if os.path.isfile(os.path.join(candidate, 'T-CR1-CAL01_온도.csv')):
        DATA_DIR = candidate
        break
else:
    raise FileNotFoundError('저장소의 data 폴더에 원본 CSV를 넣어 주세요.')

ROOT = Path(DATA_DIR).resolve().parent
FIG_DIR = os.path.join(ROOT, 'figures', '07_lot_anomaly_review_namelesspark', NOTEBOOK_NAME)
os.makedirs(FIG_DIR, exist_ok=True)
TEMP_CSV = os.path.join(DATA_DIR, 'T-CR1-CAL01_온도.csv')
EVENT_CSV = os.path.join(DATA_DIR, 'G-02_조업이벤트.csv')
REPAIR_CSV = os.path.join(DATA_DIR, 'G-01_정기수리캘린더.csv')
PART_CSV = os.path.join(DATA_DIR, 'G-01_부품마스터.csv')
MAINT_CSV = os.path.join(DATA_DIR, 'G-02_정비이력.csv')
PARTCHG_CSV = os.path.join(DATA_DIR, 'G-02_부품교체이력.csv')
ENC = 'utf-8-sig'

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK KR']:
    if cand in installed:
        rcParams['font.family'] = cand
        break
else:
    print('경고: 한글 폰트를 찾지 못했습니다.')
rcParams['axes.unicode_minus'] = False
rcParams['figure.dpi'] = 110
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False

## 원자료에서 LOT와 강종, 유효 1분 관측 만들기

LOT 시작 시각과 CR1 강종변경 시각을 연결한다. 센서별 관측 수를 따로 세어, 결측이 많은 센서의 분을 잘못 유효 처리하지 않는다.

In [ ]:
raw = pd.read_csv(TEMP_CSV, encoding=ENC, parse_dates=['MEAS_DT'])
events = pd.read_csv(EVENT_CSV, encoding=ENC, parse_dates=['EVT_DT'])
maintenance = pd.read_csv(MAINT_CSV, encoding=ENC, parse_dates=['DETC_DT', 'STRT_DT', 'END_DT'])
repair = pd.read_csv(REPAIR_CSV, encoding=ENC, parse_dates=['STRT_DT', 'END_DT'])
parts = pd.read_csv(PART_CSV, encoding=ENC)
part_changes = pd.read_csv(PARTCHG_CSV, encoding=ENC)

sensor_columns = [c for c in raw.columns if c not in ['LOT_NO', 'MEAS_DT', 'CP-Z4M']]
heat_columns = ['TC-Z3', 'TC-Z4', 'TC-Z5', 'TC-Z6']
lot_bounds = raw.groupby('LOT_NO').MEAS_DT.agg(start='min', end='max')
grade_events = events.loc[events.PLANT_CD.eq('CR1') & events.EVT_TYPE.eq('강종변경')]
# 연결이 중복되면 LOT 수가 늘어나므로 일대일 대응을 확인합니다.
lot_bounds = lot_bounds.reset_index()
lot_bounds = lot_bounds.merge(
    grade_events[['EVT_DT', 'AFT_VAL']],
    left_on='start',
    right_on='EVT_DT',
    how='left',
    validate='one_to_one',
)
lot_bounds = lot_bounds.set_index('LOT_NO')
lot_bounds = lot_bounds.rename(columns={'AFT_VAL': 'grade'})

# LOT별 1분 칸을 유지합니다. 관측 30개 미만인 센서 값은 비워 둡니다.
minute_by_lot = {}
count_by_lot = {}
for lot_no, group in raw.groupby('LOT_NO', sort=True):
    series = group.sort_values('MEAS_DT').set_index('MEAS_DT')
    means = series[sensor_columns].resample('1min').mean()
    counts = series[sensor_columns].resample('1min').count()
    means = means.mask(counts.lt(30))
    means['HEAT_MEAN'] = means[heat_columns].mean(axis=1, skipna=False)
    minute_by_lot[lot_no] = means
    count_by_lot[lot_no] = counts

raw_lot = raw.loc[raw.LOT_NO.eq(LOT)].sort_values('MEAS_DT').set_index('MEAS_DT')
minute = minute_by_lot[LOT]
lot_start = lot_bounds.loc[LOT, 'start']
lot_end = lot_bounds.loc[LOT, 'end']
grade = lot_bounds.loc[LOT, 'grade']
same_grade_lots = lot_bounds.index[lot_bounds.grade.eq(grade)].tolist()

## 이 검토에 실제 사용한 관측 수

| 센서 | 원시 유효 관측 | 원시 결측 | 전체 1분 칸 | 30개 이상인 분 | 30개 미만인 분 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 12559 | 0 | 217 | 217 | 0 |
| TC-Z4 | 12557 | 2 | 217 | 217 | 0 |
| OP-Z4 | 12559 | 0 | 217 | 217 | 0 |
| TC-Z7 | 12557 | 2 | 217 | 217 | 0 |
| TC-Z8 | 12559 | 0 | 217 | 217 | 0 |

아래 범위 표는 비교의 배경이다. 센서 단위가 다르므로 숫자의 크기를 센서끼리 직접 비교하지 않는다.

| 센서 | 1분 최솟값 | 1분 중앙값 | 1분 최댓값 | 원시 최솟값 | 원시 최댓값 |
| --- | --- | --- | --- | --- | --- |
| CP-Z4 | 0.098052 | 0.448215 | 0.46175 | 0.096911 | 0.465481 |
| TC-Z4 | 815.233 | 860.363 | 862.05 | 814.692 | 862.121 |
| OP-Z4 | 1.494 | 54.99 | 58.448 | 0 | 60.6009 |
| TC-Z7 | 309.273 | 326.183 | 326.415 | 309.071 | 326.469 |
| TC-Z8 | 308.856 | 326.059 | 326.14 | 308.625 | 326.2 |

In [ ]:
sampling_rows = []

def readable_table(table):
    # CP의 작은 변화는 보존하고 온도·출력 표의 불필요한 소수 자릿수는 줄입니다.
    shown = table.astype(object).copy()
    for row_index, row in table.iterrows():
        sensor = str(row.get('센서', ''))
        for column, value in row.items():
            if isinstance(value, (float, np.floating)) and pd.notna(value):
                digits = 3
                if sensor.startswith('CP-'):
                    digits = 6
                elif sensor.startswith('OP-') and str(column).startswith('원시'):
                    digits = 4
                shown.at[row_index, column] = f'{value:.{digits}f}'.rstrip('0').rstrip('.')
    return shown

focus_rows = []
for sensor in FOCUS:
    source = raw_lot[sensor]
    means = minute[sensor]
    counts = count_by_lot[LOT][sensor]
    sampling_rows.append({
        '센서': sensor,
        '원시 유효 관측': int(source.notna().sum()),
        '원시 결측': int(source.isna().sum()),
        '전체 1분 칸': len(means),
        '30개 이상인 분': int(counts.ge(30).sum()),
        '30개 미만인 분': int(counts.lt(30).sum()),
    })
    focus_rows.append({
        '센서': sensor,
        '1분 최솟값': means.min(),
        '1분 중앙값': means.median(),
        '1분 최댓값': means.max(),
        '원시 최솟값': source.min(),
        '원시 최댓값': source.max(),
    })
sampling_table = pd.DataFrame(sampling_rows)
focus_table = pd.DataFrame(focus_rows)
display(sampling_table)
display(readable_table(focus_table))

## 같은 강종과 실제 정비 기록의 비교 준비

아래 비교값은 관측한 자료를 요약한 수치다. 미래 자료를 포함할 수 있으므로 학습 입력의 기준값으로 그대로 사용하지 않는다.

In [ ]:
# 같은 강종도 LOT별로 보여 줍니다. 긴 LOT에 가중된 강종 평균은 만들지 않습니다.
comparison_rows = []
comparison_values = {}
for lot_no in same_grade_lots:
    data = minute_by_lot[lot_no]
    scope = '관측 전체'
    if grade == 'SPCEN' and lot_no == 240114:
        data = data.loc[data.index < pd.Timestamp('2024-11-04 20:10')]
        scope = '20:10 이전: 후반 하강 전'
    comparison_values[lot_no] = data
    for sensor in COMPARE:
        series = data[sensor].dropna()
        comparison_rows.append({
            'LOT': lot_no,
            '시작일': lot_bounds.loc[lot_no, 'start'].strftime('%Y-%m-%d'),
            '비교 범위': scope,
            '센서': sensor,
            '유효 분': len(series),
            '25%': series.quantile(.25),
            '중앙값': series.median(),
            '75%': series.quantile(.75),
        })
comparison_table = pd.DataFrame(comparison_rows)

In [ ]:
# 같은 설비의 실제 기록만 연결합니다. 가까운 기록도 원인이라는 뜻은 아닙니다.
cal = maintenance.loc[maintenance.EQP_TAG.eq('P1-CR1-CAL01')].copy()
detection_inside = cal.DETC_DT.between(lot_start, lot_end)
work_overlaps = cal.STRT_DT.le(lot_end) & cal.END_DT.ge(lot_start)
related = cal.loc[detection_inside | work_overlaps].copy()
previous = cal.loc[cal.DETC_DT.lt(lot_start)].sort_values('DETC_DT').tail(1)
following = cal.loc[cal.DETC_DT.gt(lot_end)].sort_values('DETC_DT').head(1)
previous_work = cal.loc[cal.END_DT.lt(lot_start)].sort_values('END_DT').tail(1)
following_work = cal.loc[cal.STRT_DT.gt(lot_end)].sort_values('STRT_DT').head(1)
context = pd.concat([previous, previous_work, related, following, following_work])
context = context.drop_duplicates('WO_NO').sort_values('DETC_DT')
maintenance_names = {
    'TBM': '정기점검(TBM)',
    'BM': '고장 후 정비(BM)',
    'CBM': '상태기반 정비(CBM)',
}
change_names = part_changes.merge(
    parts[['PART_CD', 'PART_NM']],
    on='PART_CD',
    how='left',
    validate='many_to_one',
)
context_rows = []
for _, record in context.iterrows():
    if pd.isna(record.DETC_DT):
        relation = '감지 시각 없음'
    elif record.DETC_DT < lot_start:
        relation = f'LOT 시작 {((lot_start-record.DETC_DT).total_seconds()/3600):.2f}시간 전 감지'
    elif record.DETC_DT > lot_end:
        relation = f'LOT 종료 {((record.DETC_DT-lot_end).total_seconds()/3600):.2f}시간 후 감지'
    else:
        relation = '관측 중 감지'
    if record.STRT_DT <= lot_end and record.END_DT >= lot_start:
        relation += '; 작업 시간도 관측과 겹침'
    elif record.END_DT < lot_start:
        relation += f'; 작업 끝은 시작 {(lot_start-record.END_DT).total_seconds()/3600:.2f}시간 전'
    elif record.STRT_DT > lot_end:
        relation += f'; 작업 시작은 종료 {(record.STRT_DT-lot_end).total_seconds()/3600:.2f}시간 후'
    changes = change_names.loc[change_names.WO_NO.eq(record.WO_NO)]
    items = []
    for _, change in changes.iterrows():
        items.append(f'{change.PART_NM} {change.QTY}개({change.CHG_RSN})')
    context_rows.append({
        '작업번호': record.WO_NO,
        '유형': maintenance_names.get(record.MNT_TYPE, record.MNT_TYPE),
        '감지': record.DETC_DT,
        '작업 시작': record.STRT_DT,
        '작업 끝': record.END_DT,
        '관측과의 관계': relation,
        '기록 내용': record.REMARK if pd.notna(record.REMARK) else '기재 없음',
        '교체': ', '.join(items) if items else '교체 행 없음',
    })
context_table = pd.DataFrame(context_rows)
lot_events = events.loc[events.PLANT_CD.eq('CR1') & events.EVT_DT.between(lot_start, lot_end)]
event_table = lot_events[['EVT_ID', 'EVT_DT', 'EVT_TYPE', 'BEF_VAL', 'AFT_VAL']].copy()
cal_repairs = repair.loc[repair.PLANT_CD.eq('CR1')].copy()
# 종료 당일을 포함합니다. 관측 중 겹침과 가까운 계획을 구분합니다.
repair_overlaps = cal_repairs.STRT_DT.le(lot_end) & (cal_repairs.END_DT+pd.Timedelta(days=1)).gt(lot_start)

markers = []
for _, record in related.iterrows():
    if lot_start <= record.DETC_DT <= lot_end:
        label = maintenance_names.get(record.MNT_TYPE, record.MNT_TYPE)
        markers.append((record.DETC_DT, f'{label} 감지', '#a22b45'))

## 질문에 답하기 위한 실제 수치

| 확인 항목 | 재계산 결과 | 해석할 때 주의 |
| --- | --- | --- |
| CP<0.40: 첫 분 / 끝까지 지속 | 07:33 / 07:43 | 일시 저하와 지속 저하를 구분 |
| 가열 4센서 평균: 지속 하강 기준 | 08:14, LOT 중앙값−3℃ = 857.418℃ | 사후 비교 규칙이며 조작 명령 시각 아님 |
| 시각 차이 | 31분 | CP 조건과 온도 조건 사이의 차이 |
| 첫/마지막 유효 5분 중앙값 | CP 0.44916→0.10121; OP4 56.720→8.088% | 원시 순간값이 아닌 1분 평균들의 중앙값 |

**관찰:** CP가 먼저 지속적으로 낮아지고, 가열·과시효 온도는 뒤이어 내려간다. 출력은 같은 시각부터 단조 감소하지 않으며 유지되거나 더 오른 뒤 말기에 크게 줄어든다. CP 기준과 구역별 온도 기준을 하나의 시작 시각으로 합치면 이러한 선후 관계가 사라진다.

**가능한 해석과 한계:** 정기수리 전 운전 단계 변화로 볼 가능성은 있지만 조업이벤트에는 정지 준비 명령이 없다. 이후 오일씰·O링 교체도 CP 변화나 누설의 원인을 입증하지 않는다. DP590 비교 LOT는 하나뿐이다.

In [ ]:
def median(lot_no, channel, start=None, end=None):
    data = minute_by_lot[lot_no]
    if start is not None:
        data = data.loc[data.index >= pd.Timestamp(start)]
    if end is not None:
        data = data.loc[data.index < pd.Timestamp(end)]
    return data[channel].median()

def at(channel, timestamp):
    return minute.loc[pd.Timestamp(timestamp), channel]

def raw_extreme(channel, maximum=True):
    series = raw_lot[channel].dropna()
    timestamp = series.idxmax() if maximum else series.idxmin()
    return timestamp, series.loc[timestamp]

def persistent_start(condition):
    # 유효 분만 넘겨 받아 마지막 유효 분까지 유지된 조건을 찾습니다.
    # 관측 종료 뒤는 알 수 없으며, 실시간 판단 규칙이 아닙니다.
    condition = condition.fillna(False)
    last_run = condition.iloc[::-1].cummin().iloc[::-1]
    times = last_run.index[last_run]
    return times[0] if len(times) else pd.NaT

cp = minute['CP-Z4'].dropna()
heat = minute['HEAT_MEAN'].dropna()
first_cp = cp.index[cp.lt(.40)][0]
cp_start = persistent_start(cp.lt(.40))
heat_limit = heat.median() - 3
heat_start = persistent_start(heat.lt(heat_limit))
evidence_rows = [
    ('CP<0.40: 첫 분 / 끝까지 지속', f'{first_cp:%H:%M} / {cp_start:%H:%M}', '일시 저하와 지속 저하를 구분'),
    ('가열 4센서 평균: 지속 하강 기준', f'{heat_start:%H:%M}, LOT 중앙값−3℃ = {heat_limit:.3f}℃', '사후 비교 규칙이며 조작 명령 시각 아님'),
    ('시각 차이', f'{(heat_start-cp_start).total_seconds()/60:.0f}분', 'CP 조건과 온도 조건 사이의 차이'),
    (
        '첫/마지막 유효 5분 중앙값',
        (
            f"CP {cp.iloc[:5].median():.5f}→{cp.iloc[-5:].median():.5f}"
            f"; OP4 {minute['OP-Z4'].dropna().iloc[:5].median():.3f}"
            f"→{minute['OP-Z4'].dropna().iloc[-5:].median():.3f}%"
        ),
        '원시 순간값이 아닌 1분 평균들의 중앙값',
    ),
]
markers.extend([
    (cp_start, 'CP가 계속 낮게 유지된 첫 분', '#a57618'),
    (heat_start, '가열 온도가 계속 낮아진 첫 분', '#9b425e'),
])

evidence_table = pd.DataFrame(evidence_rows, columns=['확인 항목', '재계산 결과', '해석할 때 주의'])
display(evidence_table)
for title, table in extra_tables:
    print(title)
    display(readable_table(table))

## 다른 센서에서도 확인한 변화

가열 센서만 보면 과시효 구역의 온도 감소와 출력 변화를 놓칠 수 있다. 온도는 내려가지만 출력은 한동안 유지되거나 상승한 뒤 말기에 크게 줄어든다. 첫·끝 구간의 차이가 음수라는 사실만으로 그 사이가 단조 하강이라고 해석하지 않는다. 이 표 자체가 네 센서를 같은 시각부터 제외하는 규칙은 아니며, 최신 05의 구역별 사후 선택은 마지막 판단에 구분해 적었다.

| 센서 | 첫 15분 유효 분 | 끝 15분 유효 분 | 첫 15분 중앙값 | 끝 15분 중앙값 | 끝−처음 | 끝 15분 최저 | 끝 15분 최고 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 15 | 15 | 326.206 | 312.102 | -14.104 | 309.273 | 314.836 |
| TC-Z8 | 15 | 15 | 326.063 | 311.643 | -14.42 | 308.856 | 314.398 |
| OP-Z7 | 15 | 15 | 65.423 | 41.875 | -23.548 | 2.287 | 78.657 |
| OP-Z8 | 15 | 15 | 54.576 | 34.301 | -20.275 | 1.879 | 63.881 |

온도는 ℃, 출력은 % 가정이다. 첫·끝 15분은 시간 칸 기준이며 센서 결측으로 유효 분 수가 다를 수 있다.

In [ ]:
RELATED_SENSORS = ['TC-Z7', 'TC-Z8', 'OP-Z7', 'OP-Z8']

related_rows = []
for sensor in RELATED_SENSORS:
    values = minute[sensor]
    first = values.iloc[:15].dropna()
    last = values.iloc[-15:].dropna()
    related_rows.append({
        '센서': sensor,
        '첫 15분 유효 분': len(first),
        '끝 15분 유효 분': len(last),
        '첫 15분 중앙값': first.median(),
        '끝 15분 중앙값': last.median(),
        '끝−처음': last.median() - first.median(),
        '끝 15분 최저': last.min(),
        '끝 15분 최고': last.max(),
    })
related_table = pd.DataFrame(related_rows)
display(readable_table(related_table))

## 전체 관측에서 이 현상이 차지하는 위치

CP가 먼저 지속적으로 낮아지고, 가열·과시효 온도는 뒤이어 내려간다. 출력은 같은 시각부터 단조 감소하지 않으며 유지되거나 더 오른 뒤 말기에 크게 줄어든다. CP 기준과 구역별 온도 기준을 하나의 시작 시각으로 합치면 이러한 선후 관계가 사라진다.

옅은 선은 원시 관측, 진한 선은 유효 1분 평균이다. 노란 영역은 다음 그림에서 확대할 범위이며 학습 제외 표시가 아니다. 다른 단위의 센서는 축을 분리했다.

![전체 관측](../../figures/07_lot_anomaly_review_namelesspark/06_LOT_240058/01_overview.png)

In [ ]:
def sensor_unit(sensor):
    if sensor.startswith('CP-'):
        return '단위 미확인'
    if sensor.startswith('OP-'):
        return '% (가정)'
    return '℃ (가정)'

def draw_timeseries(start, end, filename, title, overview=False):
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    fig, axes = plt.subplots(len(FOCUS), 1, figsize=(12, 2.05*len(FOCUS)+1.0), sharex=True)
    axes = np.atleast_1d(axes)
    source = raw_lot.loc[start:end]
    averages = minute.loc[start.floor('min'):end.ceil('min')]
    # 없는 초를 NaN으로 둬 원시 관측 공백도 선으로 이어 붙이지 않습니다.
    grid = pd.date_range(start.ceil('s'), end.floor('s'), freq='1s')
    source = source.reindex(grid)
    for ax, sensor in zip(axes, FOCUS):
        ax.plot(source.index, source[sensor], color='#9bbbd1', lw=.6, alpha=.55, label='원시 관측')
        # 표의 13:29는 13:29:00~59 평균입니다. 곡선은 그 분의 가운데에 표시합니다.
        average_times = averages.index + pd.Timedelta(seconds=30)
        ax.plot(average_times, averages[sensor], color='#153f69', lw=1.2, label='1분 평균 (분 가운데, 관측≥30)')
        if overview:
            ax.axvspan(pd.Timestamp(ZOOM_START), pd.Timestamp(ZOOM_END), color='#e3bd65', alpha=.18, label='아래에서 확대할 구간')
        for timestamp, label, color in markers:
            if '1분 최고' in label:
                timestamp = timestamp + pd.Timedelta(seconds=30)
            if start <= timestamp <= end:
                ax.axvline(timestamp, color=color, lw=.9, ls='--', alpha=.85, label=label)
        for _, record in related.iterrows():
            left = max(start, record.STRT_DT)
            right = min(end, record.END_DT)
            if left < right:
                label = maintenance_names.get(record.MNT_TYPE, record.MNT_TYPE)
                ax.axvspan(left, right, color='#b5a5cc', alpha=.13, label=f'{label} 작업')
        ax.set_ylabel(f'{sensor}\n{sensor_unit(sensor)}', fontsize=10)
        ax.tick_params(labelsize=9)
        ax.grid(alpha=.18, lw=.5)
        ax.set_xlim(start, end)
    handles, labels = axes[0].get_legend_handles_labels()
    unique = {}
    for handle, label in zip(handles, labels):
        if label not in unique:
            unique[label] = handle
    fig.legend(unique.values(), unique.keys(), loc='upper center', bbox_to_anchor=(.5, .955), ncol=3, fontsize=9, frameon=False)
    locator = mdates.AutoDateLocator(minticks=4, maxticks=9)
    axes[-1].xaxis.set_major_locator(locator)
    axes[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    axes[-1].set_xlabel('관측 시각 — 색칠은 검토 범위이며 자동 학습 제외 표시가 아닙니다.', fontsize=9)
    fig.suptitle(f'LOT {LOT} · {grade} | {PLOT_TITLE}\n{title}', fontsize=13, y=.995)
    fig.subplots_adjust(left=.115, right=.98, bottom=.075, top=.89, hspace=.28)
    figure_path = Path(FIG_DIR) / filename
    fig.savefig(figure_path, dpi=160, facecolor='white')
    plt.show()
    plt.close(fig)
    return figure_path

def draw_comparison():
    if len(same_grade_lots) < 2:
        print(f'{grade}은 관측 LOT가 하나여서 같은 강종의 독립 비교가 불가능합니다.')
        return None
    fig, axes = plt.subplots(len(COMPARE), 1, figsize=(10, 3.0*len(COMPARE)), squeeze=False)
    labels = []
    for lot_no in same_grade_lots:
        label = str(lot_no)
        if grade == 'SPCEN' and lot_no == 240114:
            label += '\n20:10 하강 전'
        labels.append(label)
    for ax, sensor in zip(axes[:, 0], COMPARE):
        values = [comparison_values[lot_no][sensor].dropna().to_numpy() for lot_no in same_grade_lots]
        boxes = ax.boxplot(values, tick_labels=labels, showfliers=False, widths=.45, patch_artist=True)
        for patch, lot_no in zip(boxes['boxes'], same_grade_lots):
            patch.set_facecolor('#82afca' if lot_no == LOT else '#dce6ec')
        ax.set_ylabel(f'{sensor}\n{sensor_unit(sensor)}')
        ax.grid(axis='y', alpha=.2)
        ax.set_title(f'{sensor}: LOT별 분포 (상자 25~75%, 가운데 선 중앙값)', fontsize=10)
    axes[-1, 0].set_xlabel('LOT — 극단점 생략. 원시 최댓값은 표·시계열에서 확인하세요.\n서로 다른 날짜·관측 길이·운전 조건을 포함합니다.')
    fig.suptitle(f'{grade} {len(same_grade_lots)}개 LOT 비교 · 진한 상자 = {LOT}', fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, .95])
    path = Path(FIG_DIR) / '03_same_grade.png'
    fig.savefig(path, dpi=160, facecolor='white')
    plt.show()
    plt.close(fig)
    return path

overview_path = draw_timeseries(lot_start, lot_end, '01_overview.png', '전체 관측과 검토 구간', overview=True)

## 원시값까지 확대해서 확인

확대 범위: **2024-06-09 07:25 ~ 2024-06-09 09:02:53**. 이 범위에 감지·작업 기록이 있으면 감지 시각은 점선, 작업 시간은 보라색으로 표시한다. 별도로 비교한 시각도 점선으로 표시하며 뜻은 범례에 적었다.

정기수리 전 운전 단계 변화로 볼 가능성은 있지만 조업이벤트에는 정지 준비 명령이 없다. 이후 오일씰·O링 교체도 CP 변화나 누설의 원인을 입증하지 않는다. DP590 비교 LOT는 하나뿐이다.

![확대 관측](../../figures/07_lot_anomaly_review_namelesspark/06_LOT_240058/02_zoom.png)

In [ ]:
zoom_path = draw_timeseries(ZOOM_START, ZOOM_END, '02_zoom.png', '관심 구간의 원시 관측과 1분 평균')

## 추가 검토: 후반에는 냉각 온도의 봉우리·골과 TC-Z9도 낮아진다

가열·과시효 4개 센서뿐 아니라 TC-OUT1·2와 TC-Z9를 따로 확인한다. 냉각 온도는 앞부분에도 반복되므로 처음과 끝의 값만 비교하지 않고 이전 파형의 봉우리·골과 비교한다.

| 센서 | 비교점 | 이전 시각 | 이전 온도℃ | 후반 시각 | 후반 온도℃ | 후반−이전℃ |
| --- | --- | --- | --- | --- | --- | --- |
| TC-OUT1 | 봉우리 | 08:09 | 290.786 | 08:50 | 280.893 | -9.893 |
| TC-OUT2 | 골 | 08:02 | 266.482 | 08:45 | 257.636 | -8.846 |
| TC-Z9 | 08:14 전/후 최저 | 05:26 | 317.441 | 08:40 | 309.978 | -7.463 |

모든 수치는 센서별 관측≥30인 1분 평균이다. 봉우리·골은 코드에 적힌 이전/후반 비교 범위 안에서 찾았으며 자동 사건 탐지 기준이 아니다. 08:14 선은 기존 가열 온도 감소의 비교 기준으로, 냉각·TC-Z9 변화의 확정 시작 시각은 아니다.

냉각 파형이 후반에 낮아지는 모습은 114에서도 관측된다. DP590은 같은 강종의 다른 LOT가 없어 강종 효과를 분리할 수 없다. TC-Z9가 반복된다는 이유로 후반 변화를 무시하거나, 주변 구역의 제외 시각을 그대로 복사하지 않는다. 세 센서는 추가 검토 대상이며 05의 기본 적용표는 바꾸지 않는다.

In [ ]:
SECONDARY_START = '2024-06-09 05:26'
SECONDARY_END = '2024-06-09 09:02:53'
SECONDARY_TITLE = '후반에는 냉각 온도의 봉우리·골과 TC-Z9도 낮아진다'
SECONDARY_KIND = 'cooling'
SECONDARY_SENSORS = ['TC-OUT1', 'TC-OUT2', 'TC-Z9']

SECONDARY_BOUNDARY = '2024-06-09 08:14'
secondary_comparisons = [
    ('TC-OUT1', '봉우리', ('2024-06-09 08:00', '2024-06-09 08:18'), ('2024-06-09 08:40', '2024-06-09 08:58')),
    ('TC-OUT2', '골', ('2024-06-09 07:50', '2024-06-09 08:10'), ('2024-06-09 08:35', '2024-06-09 08:55')),
]

# 이전 파형과 후반 파형의 국소 봉우리·골을 같은 방법으로 계산합니다.
secondary_start = pd.Timestamp(SECONDARY_START)
secondary_end = pd.Timestamp(SECONDARY_END)
secondary_boundary = pd.Timestamp(SECONDARY_BOUNDARY)

def turning_point(sensor, start, end, kind):
    values = minute.loc[start:end, sensor].dropna()
    timestamp = values.idxmax() if kind == '봉우리' else values.idxmin()
    return timestamp, values.loc[timestamp]

secondary_rows = []
secondary_points = {}
for sensor, kind, earlier, later in secondary_comparisons:
    earlier_time, earlier_value = turning_point(sensor, earlier[0], earlier[1], kind)
    later_time, later_value = turning_point(sensor, later[0], later[1], kind)
    secondary_rows.append({
        '센서': sensor,
        '비교점': kind,
        '이전 시각': f'{earlier_time:%H:%M}',
        '이전 온도℃': earlier_value,
        '후반 시각': f'{later_time:%H:%M}',
        '후반 온도℃': later_value,
        '후반−이전℃': later_value - earlier_value,
    })
    secondary_points[sensor] = [earlier_time, later_time]

# TC-Z9는 앞/뒤 최솟값 비교입니다. 두 점을 같은 파형 위치로 간주하지 않습니다.
secondary_before = minute.loc[minute.index < secondary_boundary, 'TC-Z9'].dropna()
secondary_after = minute.loc[minute.index >= secondary_boundary, 'TC-Z9'].dropna()
secondary_before_time = secondary_before.idxmin()
secondary_after_time = secondary_after.idxmin()
secondary_rows.append({
    '센서': 'TC-Z9',
    '비교점': '08:14 전/후 최저',
    '이전 시각': f'{secondary_before_time:%H:%M}',
    '이전 온도℃': secondary_before.min(),
    '후반 시각': f'{secondary_after_time:%H:%M}',
    '후반 온도℃': secondary_after.min(),
    '후반−이전℃': secondary_after.min() - secondary_before.min(),
})
secondary_points['TC-Z9'] = [secondary_before_time, secondary_after_time]

secondary_table = pd.DataFrame(secondary_rows)
display(secondary_table.round(3))


In [ ]:
# 원시 공백은 선으로 잇지 않고, 평균은 분 가운데에 표시합니다.
secondary_grid = pd.date_range(secondary_start.ceil('s'), secondary_end.floor('s'), freq='1s')
secondary_plot_raw = raw_lot.loc[secondary_start:secondary_end].reindex(secondary_grid)
secondary_plot_minute = minute.loc[secondary_start.floor('min'):secondary_end.ceil('min')]
secondary_average_times = secondary_plot_minute.index + pd.Timedelta(seconds=30)
secondary_fig, secondary_axes = plt.subplots(
    len(SECONDARY_SENSORS),
    1,
    figsize=(12, 2.35 * len(SECONDARY_SENSORS) + 1.4),
    sharex=True,
)
for ax, sensor in zip(secondary_axes, SECONDARY_SENSORS):
    ax.plot(secondary_plot_raw.index, secondary_plot_raw[sensor], color='#9bbbd1', lw=0.7, alpha=0.65, label='원시 관측')
    ax.plot(secondary_average_times, secondary_plot_minute[sensor], color='#153f69', lw=1.25, label='1분 평균 (분 가운데, 관측≥30)')
    points = secondary_points.get(sensor, [])
    if points:
        point_times = pd.DatetimeIndex(points)
        point_values = minute.loc[point_times, sensor]
        ax.scatter(point_times + pd.Timedelta(seconds=30), point_values, color='#c76f2c', s=28, zorder=5, label='위 표의 비교점')
    if SECONDARY_KIND == 'output':
        ax.axvspan(secondary_zero_run.start, secondary_zero_run.end, color='#67a7a4', alpha=0.17, label='OP-Z3 0 연속 기록 구간')
        ax.axvspan(secondary_upper_run.start, secondary_upper_run.end, color='#e5ad57', alpha=0.2, label='OP-Z3 높은 값 연속 기록 구간')
        if secondary_change_time is not None:
            ax.axvline(secondary_change_time, color='#9564a7', lw=1, ls='--', label='이전 관측 상단을 처음 넘은 시각')
    else:
        ax.axvline(secondary_boundary, color='#9564a7', lw=1, ls='--', label=f'가열 온도 비교 기준 {secondary_boundary:%H:%M}')
    unit = '% (가정)' if sensor.startswith('OP-') else '℃ (가정)'
    ax.set_ylabel(f'{sensor}\n{unit}', fontsize=10)
    ax.set_xlim(secondary_start, secondary_end)
    ax.grid(alpha=0.2, lw=0.5)
    ax.tick_params(labelsize=9)

secondary_handles = {}
for ax in secondary_axes:
    handles, labels = ax.get_legend_handles_labels()
    for handle, label in zip(handles, labels):
        if label not in secondary_handles:
            secondary_handles[label] = handle
secondary_fig.legend(
    secondary_handles.values(),
    secondary_handles.keys(),
    loc='upper center',
    bbox_to_anchor=(0.5, 0.92),
    ncol=3,
    fontsize=8.5,
    frameon=False,
)
secondary_locator = mdates.AutoDateLocator(minticks=5, maxticks=9)
secondary_axes[-1].xaxis.set_major_locator(secondary_locator)
secondary_axes[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(secondary_locator))
secondary_axes[-1].set_xlabel('관측 시각 — 점·음영·기준선은 비교용이며 자동 제외 표시가 아닙니다.', fontsize=9)
secondary_fig.suptitle(f'LOT {LOT} · {grade}\n{SECONDARY_TITLE}', fontsize=13, y=0.995)
secondary_fig.subplots_adjust(left=0.105, right=0.985, bottom=0.1, top=0.80, hspace=0.3)
secondary_path = Path(FIG_DIR) / '04_secondary_review.png'
secondary_fig.savefig(secondary_path, dpi=160, facecolor='white')
plt.show()
plt.close(secondary_fig)


![후반에는 냉각 온도의 봉우리·골과 TC-Z9도 낮아진다](../../figures/07_lot_anomaly_review_namelesspark/06_LOT_240058/04_secondary_review.png)

옅은 선은 원시 관측, 진한 선은 1분 평균이다. 표는 분 시작 시각을 쓰고 평균 곡선·비교점은 30초 뒤인 분 가운데에 표시한다. 관측 공백은 보간하지 않는다.

## 추가 검토: 말기에는 건조 출력 두 개도 낮아지고 마지막 원시값은 0이다

가열·과시효와 냉각만 보면 건조 구역의 말기 변화를 놓칠 수 있다. **08:40:00~09:02:53**을 따로 살펴보고, 아래 표에서는 **08:50~09:02의 1분 평균 13개**를 비교한다. 감소·상승 횟수는 이웃한 분 사이의 12번 비교다.

| 센서 | 08:50 1분 평균 | 09:02 1분 평균 | 감소 / 상승 횟수 | 09:02:53 원시값 |
| --- | --- | --- | --- | --- |
| TC-Z1 | 97.734℃ | 96.160℃ | 7 / 5 | 96.172℃ |
| TC-Z2 | 98.424℃ | 95.532℃ | 12 / 0 | 95.531℃ |
| OP-Z1 | 63.582% | 1.938% | 12 / 0 | 0.000% |
| OP-Z2 | 10.530% | 0.520% | 12 / 0 | 0.000% |

**관찰:** 두 출력과 TC-Z2의 1분 평균은 이 비교 구간에서 12번 연속 낮아진다. TC-Z1은 중간에 5번 상승하므로 네 센서가 모두 단조 하강한 것은 아니다. OP-Z1은 앞서 08:47의 59.012%에서 08:50의 63.582%로 반등했다. 08:40부터 계속 하강했다고 해석하지 않는다. 온도·출력 단위는 데이터사전의 가정이다.

확인한 범위의 원시 행은 **1,328개**이고 네 센서 모두 결측은 0개다. 마지막 09:02분에도 각 센서 관측이 **53개** 있어 30개 기준을 만족한다. OP-Z1·2의 원시 0은 각각 1개이며 모두 09:02:53의 마지막 기록이다. 마지막 1분 평균이 0이라는 뜻은 아니다.

**해석과 학습 판단:** 08:40은 선택한 검토 범위의 시작이며 현상 시작이나 정지 명령 시각이 아니다. 다른 DP590 LOT가 없어 같은 강종에서 반복되는지 비교할 수 없고, 이 변화만으로 고장이나 건조 중단을 확정할 수 없다. 최신 05에서는 검토 목록에만 추가하며 `TC-Z1`·`TC-Z2`·`OP-Z1`·`OP-Z2`의 기본 학습 보류 표시는 모두 0으로 유지한다.


In [ ]:
DRYING_START = pd.Timestamp('2024-06-09 08:40:00')
DRYING_END = lot_end
DRYING_COMPARE_START = pd.Timestamp('2024-06-09 08:50')
DRYING_SENSORS = ['TC-Z1', 'TC-Z2', 'OP-Z1', 'OP-Z2']

# 08:40은 살펴볼 범위의 시작이고, 아래 수치 비교는 08:50부터입니다.
drying_raw = raw_lot.loc[DRYING_START:DRYING_END, DRYING_SENSORS]
drying_means = minute.loc[DRYING_START:DRYING_END, DRYING_SENSORS]
drying_counts = count_by_lot[LOT].loc[DRYING_START:DRYING_END, DRYING_SENSORS]
drying_comparison = drying_means.loc[DRYING_COMPARE_START:]

drying_rows = []
drying_quality_rows = []
for sensor in DRYING_SENSORS:
    values = drying_comparison[sensor]
    changes = values.diff().dropna()
    drying_rows.append({
        '센서': sensor,
        '08:50 1분 평균': values.iloc[0],
        '09:02 1분 평균': values.iloc[-1],
        '감소 횟수': int(changes.lt(0).sum()),
        '상승 횟수': int(changes.gt(0).sum()),
        '09:02:53 원시값': drying_raw[sensor].iloc[-1],
    })
    drying_quality_rows.append({
        '센서': sensor,
        '08:40 이후 원시 행': len(drying_raw),
        '원시 결측': int(drying_raw[sensor].isna().sum()),
        '30개 이상인 분': int(drying_counts[sensor].ge(30).sum()),
        '마지막 분 관측': int(drying_counts[sensor].iloc[-1]),
        '원시 0 관측': int(drying_raw[sensor].eq(0).sum()),
    })

drying_table = pd.DataFrame(drying_rows)
drying_quality_table = pd.DataFrame(drying_quality_rows)
drying_rebound = minute.loc[
    ['2024-06-09 08:47', '2024-06-09 08:50'],
    ['OP-Z1'],
].copy()

display(drying_table.round(3))
display(drying_quality_table)
print('08:50 전에 OP-Z1이 반등한 두 분의 평균')
display(drying_rebound.round(3))


In [ ]:
# 네 센서를 각각 그려 온도와 출력의 단위·크기를 섞지 않습니다.
drying_grid = pd.date_range(DRYING_START, DRYING_END.floor('s'), freq='1s')
drying_plot_raw = drying_raw.reindex(drying_grid)
drying_average_times = drying_means.index + pd.Timedelta(seconds=30)
drying_fig, drying_axes = plt.subplots(
    4,
    1,
    figsize=(12, 10.2),
    sharex=True,
)
for ax, sensor in zip(drying_axes, DRYING_SENSORS):
    ax.plot(
        drying_plot_raw.index,
        drying_plot_raw[sensor],
        color='#9bbbd1',
        lw=0.7,
        alpha=0.7,
        label='원시 관측',
    )
    ax.plot(
        drying_average_times,
        drying_means[sensor],
        color='#153f69',
        lw=1.35,
        label='1분 평균 (분 가운데, 관측≥30)',
    )
    ax.axvline(
        DRYING_COMPARE_START,
        color='#9564a7',
        lw=1,
        ls='--',
        label='위 표의 비교 시작: 08:50',
    )
    if sensor.startswith('OP-'):
        ax.scatter(
            [DRYING_END],
            [drying_raw[sensor].iloc[-1]],
            color='#bd6632',
            s=30,
            zorder=5,
            clip_on=False,
            label='마지막 원시 출력: 0',
        )
        unit = '% (가정)'
    else:
        unit = '℃ (가정)'
    ax.set_ylabel(f'{sensor}\n{unit}', fontsize=10)
    ax.set_xlim(DRYING_START, DRYING_END)
    ax.grid(alpha=0.2, lw=0.5)
    ax.tick_params(labelsize=9)

drying_handles = {}
for ax in drying_axes:
    handles, labels = ax.get_legend_handles_labels()
    for handle, label in zip(handles, labels):
        if label not in drying_handles:
            drying_handles[label] = handle

drying_fig.legend(
    drying_handles.values(),
    drying_handles.keys(),
    loc='upper center',
    bbox_to_anchor=(0.5, 0.925),
    ncol=2,
    fontsize=9,
    frameon=False,
)
drying_locator = mdates.MinuteLocator(interval=4)
drying_axes[-1].xaxis.set_major_locator(drying_locator)
drying_axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
drying_axes[-1].set_xlabel(
    '2024-06-09 관측 시각 — 08:40은 살펴볼 범위의 시작이며 정지 명령 시각이 아닙니다.',
    fontsize=9,
)
drying_fig.suptitle(
    f'LOT {LOT} · {grade}\n말기에는 건조 출력 두 개가 낮아지고 마지막 원시값은 0이다',
    fontsize=13,
    y=0.985,
)
drying_fig.subplots_adjust(
    left=0.105,
    right=0.98,
    bottom=0.085,
    top=0.83,
    hspace=0.25,
)
drying_path = Path(FIG_DIR) / '05_drying_review.png'
drying_fig.savefig(drying_path, dpi=160, facecolor='white')
plt.show()
plt.close(drying_fig)


![말기에 낮아지는 건조 온도와 출력](../../figures/07_lot_anomaly_review_namelesspark/06_LOT_240058/05_drying_review.png)

네 센서를 각각의 축에 그렸다. 옅은 선은 원시 관측, 진한 선은 센서별 유효 관측 30개 이상인 1분 평균이며 분 가운데에 표시한다. 관측 공백은 보간하지 않는다. 점선은 위 표의 08:50 비교 시작, 주황 점은 마지막 원시 출력 0을 뜻하며 자동 제외 표시가 아니다.


## 같은 강종 비교

같은 강종 **1개 LOT**: 240058. 각 LOT를 따로 비교하며, 긴 LOT에 가중된 강종 전체 평균은 만들지 않았다. 날짜·관측 길이·운전 상태·소재 부하가 다르므로 강종 효과만 분리한 비교는 아니다. **독립적인 같은 강종 비교 LOT가 없으므로 이 LOT의 범위를 정상 한계로 확정할 수 없다.**

| LOT | 시작일 | 비교 범위 | 센서 | 유효 분 | 25% | 중앙값 | 75% |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 240058 | 2024-06-09 | 관측 전체 | CP-Z4 | 217 | 0.168553 | 0.448215 | 0.450297 |
| 240058 | 2024-06-09 | 관측 전체 | OP-Z4 | 217 | 50.13 | 54.99 | 56.783 |

In [ ]:
display(readable_table(comparison_table))
comparison_path = draw_comparison()

## 조업·정비 기록은 무엇을 뒷받침하는가

CAL01의 관측 중 감지·작업 중첩과 가장 가까운 이전/이후 감지·작업 기록을 구분했다. 가까운 기록을 자동 원인으로 연결하지 않는다. 정기점검(TBM), 고장 후 정비(BM), 상태기반 정비(CBM)는 기록의 유형이다. 정비이력의 감지 시각은 실제 이상 시작 시각과 같다고 볼 수 없으며, 특히 정기점검(TBM)의 감지를 고장 발생으로 해석하지 않는다. 정기수리 캘린더는 계획 기간이며, 실제 작업 시각은 별도의 정비이력과 구분한다.

| 작업번호 | 유형 | 감지 | 작업 시작 | 작업 끝 | 관측과의 관계 | 기록 내용 | 교체 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00833 | 고장 후 정비(BM) | 2024-05-25 15:55:00 | 2024-05-25 19:06:00 | 2024-05-26 05:01:00 | LOT 시작 349.52시간 전 감지; 작업 끝은 시작 336.42시간 전 | 1호기 재발, 확인 중 | 교체 행 없음 |
| WO-2024-00916 | 정기점검(TBM) | 2024-06-10 06:23:00 | 2024-06-10 08:45:00 | 2024-06-10 10:48:00 | LOT 종료 21.34시간 후 감지; 작업 시작은 종료 23.70시간 후 | 기재 없음 | 오일씰 150x190x18 2개(수명도래), O링 세트 1개(손상) |

관측 안에 있는 CR1 조업이벤트:

| EVT_ID | EVT_DT | EVT_TYPE | BEF_VAL | AFT_VAL |
| --- | --- | --- | --- | --- |
| EVT-2024-0117 | 2024-06-09 05:26:00 | 강종변경 | SGCC | DP590 |

이 CSV의 CR1 이벤트 유형은 강종변경이다. 이 기록만으로 출력 설정, 재가열·정지 명령, 소재 이동 속도를 확인할 수 없다.

관측과 겹치는 CR1 정기수리 계획은 **0건**이다. 아래 계획의 종료일은 당일까지 포함한다. 계획이 가깝다는 사실과 센서 변화 원인은 구분한다.

| PLAN_ID | STRT_DT | END_DT | SCOPE |
| --- | --- | --- | --- |
| PLAN-2024-02 | 2024-06-10 00:00:00 | 2024-06-14 00:00:00 | 연속소둔로 노내 정비 |
| PLAN-2024-04 | 2024-11-05 00:00:00 | 2024-11-10 00:00:00 | 냉연 라인 전 설비 정기수리 |

In [ ]:
display(context_table)
display(event_table)
print('관측과 겹친 CR1 정기수리 계획:', int(repair_overlaps.sum()), '건')
display(cal_repairs[['PLAN_ID', 'STRT_DT', 'END_DT', 'SCOPE']])

## 학습에 넣을지에 대한 현재 판단

최신 05 기본안은 CP와 두 온도 구역의 경계를 분리한다. 원자료 CP 이탈은 07:33:21에 시작하고, **1분 평균이 0.40 미만으로 끝까지 유지되는 조건은 07:43**부터다. 둘을 합치면 CP를 보류한 유효 분은 90분이다.

가열 8개는 `TC-Z3`~`TC-Z6`·`OP-Z3`~`OP-Z6`, 과시효 4개는 `TC-Z7`·`TC-Z8`·`OP-Z7`·`OP-Z8`이다. 온도와 출력의 관계가 달라지는 운전 단계로 보고 함께 보류한 선택이며, 출력 하나하나가 그 시각부터 계속 감소한다는 뜻은 아니다. `TC-Z9`는 자동으로 묶지 않고 검토만 한다. 07:32의 작은 CP 이탈도 검토만 한다.

**아래 시간은 시작·끝이 포함되는 1분 칸 전체**를 뜻한다. 예를 들어 04:59~05:00은 04:59:00 이상 05:01:00 미만이다. 초 단위 원자료 근거와 사후 경계는 05 적용표에서 구분한다. 원본 행을 지우지 않고 대상 센서에만 표시하며, 고장 정답이 아니다.

| 05 구간 | 대상 | 표시하는 1분 범위 | 실제 유효 보류분 |
| --- | --- | --- | --- |
| A006 | `CP-Z4` | 06/09 07:33 ~ 06/09 07:42 | 10분 |
| A007 | `CP-Z4` | 06/09 07:43 ~ 06/09 09:02 | 80분 |
| A008 | 가열 8개 | 06/09 08:14 ~ 06/09 09:02 | 센서당 49분 |
| A009 | 과시효 4개 | 06/09 08:21 ~ 06/09 09:02 | 센서당 42분 |

관찰된 차이, 가능한 원인, 모델 입력·학습 판단은 서로 구분한다. 이 노트북의 색칠과 감지선은 정답 라벨이 아니다. 단일 LOT·강종별 소수 LOT·간헐 관측이라는 한계를 유지한다.

[전체 LOT 해석(노션)](https://app.notion.com/p/3f7e609a45d080b88934e41f4403a466) · [05 학습 제외 기준](../05_exclusion_windows_namelesspark.ipynb)